In [24]:
%%capture
import os
import pprint
import pandas as pd
from convokit import Corpus, download
from pathlib import Path
from datasets import load_dataset
import janitor

hf_token = os.getenv('HF_TOKEN')

In [25]:
# load gap data from convokitA
gap = Corpus(filename = download("gap-corpus"))

convo_df = gap.get_conversations_dataframe().clean_names().reset_index(drop = False).drop(columns=['vectors'])
speaker_df = gap.get_speakers_dataframe().clean_names().reset_index(drop = False).drop(columns=['vectors'])
utt_df = gap.get_utterances_dataframe().clean_names().reset_index(drop = False).drop(columns=['vectors'])

Dataset already exists at /Users/jasminec/.convokit/saved-corpora/gap-corpus


In [26]:
ut_group_map  = (
    convo_df[['id', 'meta_group_number', 'meta_meeting_size', 'meta_meeting_length_in_minutes']]
    .rename(columns={
        'id': 'conversation_id',
        'meta_group_number': 'group_id',
        'meta_meeting_size': 'num_participants',
        'meta_meeting_length_in_minutes': 'duration'
        })
)

# merge with utterances 
utt_df =utt_df.merge(ut_group_map, on='conversation_id', how='left')

In [ ]:
# build full transcripts
transcripts = []
convo = None
unlabeled_convo = None
group_id = None

for i, r in utt_df.iterrows():
    if group_id == r['group_id']:
        convo += f"{r['speaker']}: {r['text']}\n"
        unlabeled_convo += f"{r['text']}\n"
    elif group_id != r['group_id'] and group_id is not None:
        transcripts.append(
            {
                'group_id': group_id,
                'convo': convo,
                'unlabeled_convo': unlabeled_convo,
            }
        )
        group_id = r['group_id']
        convo = f"{r['speaker']}: {r['text']}\n"
        unlabeled_convo = f"{r['text']}\n"
    else:
        group_id = r['group_id']
        convo = f"{r['speaker']}: {r['text']}\n"
        unlabeled_convo = f"{r['text']}\n"

if group_id is not None and convo is not None:
    transcripts.append(
        {
            'group_id': group_id,
            'convo': convo,
            'unlabeled_convo': unlabeled_convo,
        }
    )

In [47]:
# build chunks for evals
chunk_size = 25

meeting_size_map = (
    convo_df[["meta_group_number", "meta_meeting_size"]]
    .drop_duplicates()
    .assign(
        meta_group_number=lambda d: d["meta_group_number"].astype(str),
        meta_meeting_size=lambda d: pd.to_numeric(d["meta_meeting_size"], errors="coerce"),
    )
    .set_index("meta_group_number")["meta_meeting_size"]
    .to_dict()
)

excerpts = []
for t in transcripts:
    group_id = str(t["group_id"])

    labeled_lines = [line for line in t["convo"].splitlines() if line.strip()]
    unlabeled_lines = [line for line in t["unlabeled_convo"].splitlines() if line.strip()]

    # Fallback: if lengths ever drift, derive unlabeled from labeled
    if len(unlabeled_lines) != len(labeled_lines):
        unlabeled_lines = [
            line.split(":", 1)[1].strip() if ":" in line else line
            for line in labeled_lines
        ]

    for chunk_ix, start in enumerate(range(0, len(labeled_lines), chunk_size)):
        end = min(start + chunk_size, len(labeled_lines))
        labeled_chunk = labeled_lines[start:end]
        unlabeled_chunk = unlabeled_lines[start:end]

        speakers = sorted({line.split(":", 1)[0] for line in labeled_chunk if ":" in line})
        gt = meeting_size_map.get(group_id)
        global_speaker_count = int(gt) if pd.notna(gt) else len(speakers)

        excerpts.append(
            {
                "source": f"{group_id}:{start + 1}-{end}",
                "group_id": group_id,
                "chunk_ix": chunk_ix,
                "line_start": start + 1,
                "line_end": end,
                "num_lines": len(labeled_chunk),
                "speakers": set(speakers),
                "num_speakers": len(speakers),
                "global_speaker_count": global_speaker_count,
                "baseline_text": "\n".join(labeled_chunk) + "\n",
                "unlabeled_text": "\n".join(unlabeled_chunk) + "\n",
            }
        )

len(excerpts)

[{'source': '1:1-25',
  'group_id': '1',
  'chunk_ix': 0,
  'line_start': 1,
  'line_end': 25,
  'num_lines': 25,
  'speakers': {'1.Blue', '1.Green', '1.Pink'},
  'num_speakers': 3,
  'global_speaker_count': 3,
  'baseline_text': '1.Pink: "So what did everyone do as one"\n1.Blue: "I did uh cigarette lighter"\n1.Blue: "For one"\n1.Pink: "Mm okay I did knife"\n1.Green: "Knife"\n1.Blue: "Okay well do knife then"\n1.Blue: "$"\n1.Pink: "$"\n1.Pink: "Well I think it that and and um the cigarette lighter are all pretty important"\n1.Blue: "Yeah yeah makes sense"\n1.Blue: "Mhm"\n1.Pink: "So I would say cigarette lighter is two"\n1.Pink: "I didnt write that down but I kinda made a connection afterwards"\n1.Blue: "$ Yeah okay"\n1.Pink: "$"\n1.Green: "I did two as compass"\n1.Blue: "Compass"\n1.Pink: "Mhm thats what I did initially"\n1.Blue: "I did that as like nine"\n1.Blue: "$"\n1.Green: "Because theres no point in being able to start a fire if you cant get out right"\n1.Pink: "Well theres "\n1

In [63]:
import asyncio
import json
import re
import time
from utils.utils import send_openrouter_request_async

model_ids = [
    "qwen/qwen3-vl-8b-instruct",
    "qwen/qwen3-vl-30b-a3b-instruct",
    "qwen/qwen3-vl-32b-instruct",
    "qwen/qwen3.5-9b",
    "qwen/qwen3.5-27b",
    "mistralai/mixtral-8x22b-instruct",
    "allenai/olmo-3.1-32b-instruct",
    # "mistralai/mixtral-8x7b-instruct",
]

n_eval_per_model = min(15, len(excerpts))
concurrency = 6
temperature = 0.0
max_tokens = 4
use_labeled_text = False  # False = unlabeled_text, True = baseline_text

def parse_first_int(text):
    m = re.search(r"\b(\d+)\b", str(text))
    return int(m.group(1)) if m else None

def extract_provider_name(provider):
    if isinstance(provider, str):
        return provider
    if isinstance(provider, dict):
        return (
            provider.get("name")
            or provider.get("provider")
            or provider.get("id")
            or provider.get("slug")
            or str(provider)
        )
    return None if provider is None else str(provider)

def as_full_text_or_json(value):
    if value is None:
        return None
    if isinstance(value, str):
        return value
    if isinstance(value, list):
        text_parts = []
        for item in value:
            if isinstance(item, dict):
                if "text" in item and item["text"] is not None:
                    text_parts.append(str(item["text"]))
            elif item is not None:
                text_parts.append(str(item))
        if text_parts:
            return "".join(text_parts)
    try:
        return json.dumps(value, ensure_ascii=False)
    except TypeError:
        return str(value)

async def eval_one(model_id, example, semaphore):
    text_key = "baseline_text" if use_labeled_text else "unlabeled_text"
    prompt = f"""You will be given a conversation excerpt.
How many distinct speakers are present?
Output only a single integer.

Excerpt:
{example[text_key]}
"""

    async with semaphore:
        t0 = time.perf_counter()
        try:
            response, reason, refusal, provider = await send_openrouter_request_async(
                messages=[
                    {"role": "user", "content": prompt},
                ],
                model=model_id,
                temperature=temperature,
                max_tokens=max_tokens,
            )
            error = None
        except Exception as exc:
            response, reason, refusal, provider = "", None, None, None
            error = str(exc)
        elapsed_s = time.perf_counter() - t0

    response_text_for_parse = as_full_text_or_json(response) or ""
    pred = parse_first_int(response_text_for_parse)
    return {
        "model_id": model_id,
        "task_variant": "with_labels" if use_labeled_text else "without_labels",
        "source": example["source"],
        "group_id": example["group_id"],
        "predicted_speakers": pred,
        "global_speaker_count": example["global_speaker_count"],
        "correct": pred == example["global_speaker_count"],
        "elapsed_s": elapsed_s,
        "response": response,
        "reasoning": reason,
        "refusal": refusal,
        "provider": provider,
        "error": error,
    }

async def run_eval(examples, models, max_examples, max_concurrency):
    sem = asyncio.Semaphore(max_concurrency)
    tasks = [
        eval_one(model_id, e, sem)
        for model_id in models
        for e in examples[:max_examples]
    ]
    return await asyncio.gather(*tasks)

results = await run_eval(excerpts, model_ids, n_eval_per_model, concurrency)
results_df = pd.DataFrame(results)
results_df["provider_name"] = results_df["provider"].apply(extract_provider_name)

summary_df = (
    results_df.groupby("model_id", as_index=False)
    .agg(
        task_variant=("task_variant", "first"),
        n=("source", "count"),
        accuracy=("correct", "mean"),
        n_api_errors=("error", lambda s: s.notna().sum()),
    )
    .sort_values(["accuracy", "n_api_errors"], ascending=[False, True])
)

display(summary_df)

# Show full model outputs for debugging.
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
display(
    results_df[
        [
            "model_id",
            "provider_name",
            "provider",
            "task_variant",
            "source",
            "predicted_speakers",
            "global_speaker_count",
            "correct",
            "response",
            "reasoning",
            "refusal",
            "error",
        ]
    ].head(90)
)

,model_id,task_variant,n,accuracy,n_api_errors
1,mistralai/mixtral-8x22b-instruct,without_labels,15,0.733333,0
5,qwen/qwen3.5-27b,without_labels,15,0.333333,0
2,qwen/qwen3-vl-30b-a3b-instruct,without_labels,15,0.133333,0
3,qwen/qwen3-vl-32b-instruct,without_labels,15,0.133333,0
0,allenai/olmo-3.1-32b-instruct,without_labels,15,0.066667,0
4,qwen/qwen3-vl-8b-instruct,without_labels,15,0.066667,0
6,qwen/qwen3.5-9b,without_labels,15,0.000000,0


,model_id,provider_name,task_variant,source,predicted_speakers,global_speaker_count,correct,response,refusal,error
0,qwen/qwen3-vl-8b-instruct,AtlasCloud,without_labels,1:1-25,2.0,3,False,2,None,None
1,qwen/qwen3-vl-8b-instruct,Novita,without_labels,1:26-50,2.0,3,False,2,None,None
2,qwen/qwen3-vl-8b-instruct,Novita,without_labels,1:51-75,2.0,3,False,2,None,None
3,qwen/qwen3-vl-8b-instruct,Novita,without_labels,1:76-100,2.0,3,False,2,None,None
4,qwen/qwen3-vl-8b-instruct,Novita,without_labels,1:101-125,4.0,3,False,4,None,None
...,...,...,...,...,...,...,...,...,...,...
85,mistralai/mixtral-8x22b-instruct,Mistral,without_labels,1:251-275,3.0,3,True,3,None,None
86,mistralai/mixtral-8x22b-instruct,Mistral,without_labels,1:276-300,3.0,3,True,3,None,None
87,mistralai/mixtral-8x22b-instruct,Mistral,without_labels,1:301-325,4.0,3,False,4,None,None
88,mistralai/mixtral-8x22b-instruct,Mistral,without_labels,1:326-349,3.0,3,True,3,None,None
